In [ ]:
import os
import json
import time
import pickle
import numpy as np
from tqdm.notebook import tqdm
import torch
from transformers import AutoTokenizer, AutoModel
import faiss

# 경로 설정
DATA_PATH = "pasred_data/parsed_ctg-studies.json"
INDEX_PATH = "faiss_index" 
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-4B" 

# 배치 설정
BATCH_SIZE = 32  
MAX_LENGTH = 512

# GPU 설정
DEVICE = "cuda:1" 

def load_embedding_model():
    # 임베딩 모델 로딩
    start = time.time()
    
    tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        EMBEDDING_MODEL,
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to(DEVICE)
    model.eval()
    
    print(f" 임베딩 모델 로딩 완료 ({time.time() - start:.1f}초)")
    return tokenizer, model


def extract_searchable_text(nct_id: str, study_data: dict) -> str:
    # 임상시험 데이터에서 검색용 텍스트 추출

    overview = study_data.get("Study Overview", {})
    
    # 주요 필드 추출
    brief_title = overview.get("Brief Title", "")
    official_title = overview.get("Official Title", "")
    conditions = overview.get("Conditions", [])
    brief_summary = overview.get("Brief Summary", "")
    study_type = overview.get("Study Type", "")
    
    # Interventions 추출
    interventions = overview.get("Interventions", [])
    intervention_texts = []
    for inv in interventions:
        inv_type = inv.get("Type", "")
        inv_name = inv.get("Name", "")
        if inv_type or inv_name:
            intervention_texts.append(f"{inv_type}: {inv_name}")
    
    # 검색용 텍스트 조합
    text_parts = [
        f"Title: {brief_title}",
        f"Conditions: {', '.join(conditions) if conditions else 'N/A'}",
        f"Study Type: {study_type}",
        f"Interventions: {', '.join(intervention_texts) if intervention_texts else 'N/A'}",
        f"Summary: {brief_summary[:500]}"  
    ]
    
    return " | ".join(text_parts)


def get_embeddings(texts: list, tokenizer, model) -> np.ndarray:
    # 텍스트 리스트를 임베딩 벡터로 변환
    
    with torch.no_grad():
        inputs = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(DEVICE)
        
        outputs = model(**inputs)
        
        # Mean pooling
        attention_mask = inputs["attention_mask"]
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        
        # l2 정규화
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        
    return embeddings.cpu().numpy().astype('float32')


def build_faiss_index():

    os.makedirs(INDEX_PATH, exist_ok=True)
    
    # 모델 로딩
    tokenizer, model = load_embedding_model()
    
    # 데이터 로딩
    start = time.time()
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    nct_ids = []
    texts = []
    
    for nct_id, study_data in tqdm(data.items(), desc="텍스트 추출"):
        text = extract_searchable_text(nct_id, study_data)
        if text.strip():
            nct_ids.append(nct_id)
            texts.append(text)
    
    print(f" 텍스트 추출 개수 : {len(texts):,}")
    
    # 배치 임베딩
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="임베딩"):
        batch_texts = texts[i:i + BATCH_SIZE]
        batch_embeddings = get_embeddings(batch_texts, tokenizer, model)
        all_embeddings.append(batch_embeddings)
        
        if i % (BATCH_SIZE * 100) == 0:
            torch.cuda.empty_cache()
    
    embeddings = np.vstack(all_embeddings)
    print(f"임베딩 완료 : shape={embeddings.shape}")
    
    # FAISS 인덱스 생성
    dimension = embeddings.shape[1]
    
    # Inner Product (코사인 유사도, 정규화된 벡터)
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    
    print(f"FAISS 인덱스 생성 완료 : {index.ntotal:,}개 벡터")
    
    # FAISS 인덱스 저장
    faiss.write_index(index, os.path.join(INDEX_PATH, "clinical_trials.index"))
    
    # 메타데이터 저장 (NCT ID 리스트)
    with open(os.path.join(INDEX_PATH, "nct_ids.pkl"), 'wb') as f:
        pickle.dump(nct_ids, f)
    
    # 원본 데이터도 저장 (검색 후 Few-shot에 사용)
    with open(os.path.join(INDEX_PATH, "study_data.pkl"), 'wb') as f:
        pickle.dump(data, f)
    

    print(f"- clinical_trials.index ({index.ntotal:,} vectors)")
    print(f"- nct_ids.pkl ({len(nct_ids):,} IDs)")

    
    # GPU 메모리 해제
    del model
    del tokenizer
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    
    return index, nct_ids, data


# 실행
if __name__ == "__main__":
    total_start = time.time()
    
    index, nct_ids, data = build_faiss_index()
    